In [ ]:
from datetime import datetime
from pathlib import Path

import polars as pl

from calculate_distance_between_pings import calculate_distance

In [2]:
DIR = "/mnt/common-ssd/group-dirs/tkom-2026-project/pickwell/budapest/partitioned/"

In [11]:
test = pl.DataFrame(
    {
        "device_id": [1, 1, 1, 1, 2, 2, 2],
        "timestamp": [
            datetime(2026, 10, 19, 10, 42, 38),
            datetime(2026, 10, 19, 10, 42, 59),
            datetime(2026, 10, 19, 10, 44, 11),
            datetime(2026, 10, 19, 10, 43, 7),
            datetime(2026, 10, 18, 17, 15, 43),
            datetime(2026, 10, 18, 17, 15, 49),
            datetime(2026, 10, 18, 17, 18, 2),
        ],
        "longitude": [
            19.033973,
            19.0334969,
            19.0321977,
            19.0326872,
            19.1095298,
            19.1099686,
            19.1165223,
        ],
        "latitude": [
            47.501812,
            47.501803,
            47.501977,
            47.501878,
            47.472143,
            47.471786,
            47.470600,
        ],
    }
)

In [ ]:
time_diff = 21 + 8 + 64 + 6 + 133

# assert calculate_distance(test.sort(["device_id", "timestamp"]))['time_diff'].sum().seconds, time_diff
calculate_distance(test.sort(["device_id", "timestamp"]))[
    "time_diff"
].sum().seconds == time_diff

In [40]:
calculate_distance(test.sort(["device_id", "timestamp"])).with_columns(
    (pl.col.distance_km * 1000 / pl.col.time_diff.dt.total_seconds()).alias("speed_mps")
)

device_id,timestamp,longitude,latitude,prev_latitude,prev_longitude,current_point,previous_point,distance_km,time_diff,speed_mps
i64,datetime[μs],f64,f64,f64,f64,struct[2],struct[2],f64,duration[μs],f64
1,2026-10-19 10:42:38,19.033973,47.501812,null,null,"{47.501812,19.033973}","{null,null}",null,null,null
1,2026-10-19 10:42:59,19.033497,47.501803,47.501812,19.033973,"{47.501803,19.033497}","{47.501812,19.033973}",0.035778,21s,1.703736
1,2026-10-19 10:43:07,19.032687,47.501878,47.501803,19.033497,"{47.501878,19.032687}","{47.501803,19.033497}",0.061393,8s,7.674172
1,2026-10-19 10:44:11,19.032198,47.501977,47.501878,19.032687,"{47.501977,19.032198}","{47.501878,19.032687}",0.038383,1m 4s,0.599741
2,2026-10-18 17:15:43,19.10953,47.472143,null,null,"{47.472143,19.10953}","{null,null}",null,null,null
2,2026-10-18 17:15:49,19.109969,47.471786,47.472143,19.10953,"{47.471786,19.109969}","{47.472143,19.10953}",0.05161,6s,8.601647
2,2026-10-18 17:18:02,19.116522,47.4706,47.471786,19.109969,"{47.4706,19.116522}","{47.471786,19.109969}",0.509946,2m 13s,3.83418


In [ ]:
Path(f"{DIR}/with_distance/").mkdir(exist_ok=True, parents=True)
for i in range(128, 256):
    df = pl.read_parquet(f"{DIR}/{i:02x}/pings.parquet")
    df = calculate_distance(df)
    df = df.with_columns(
        (pl.col.distance_km * 1000 / pl.col.time_diff.dt.total_seconds()).alias(
            "speed_mps"
        )
    )
    df.drop(
        [
            "prev_latitude",
            "prev_longitude",
            "current_point",
            "previous_point",
            "id_with_type",
        ],
        strict=False,
    ).write_parquet(f"{DIR}/with_distance/{i:02x}.parquet")
# 16 runs 73s
# 112 runs 6m 24s
# 128 runs 7m 19s